# SLS² fork: Kaggle에서 Reacher 데이터 생성과 첫 학습

실행 코드: [KwangBin417/SLS-squared](https://github.com/KwangBin417/SLS-squared/tree/kwangbin/kaggle-reacher). 원본 main을 보존하고 `kwangbin/kaggle-reacher` 브랜치에서 변경 이력을 관리합니다.

**실행 전에:** [Kaggle 계정 설정](https://www.kaggle.com/settings)에서 **Phone Verification**을 완료해야 GPU를 사용할 수 있습니다. GPU 항목이 회색이면 인증 상태를 먼저 확인합니다. 인증 후 노트북을 새로고침하고 Settings → Accelerator → GPU T4 ×2를 선택한 뒤, Session options의 Internet을 On으로 설정합니다. `kernelSessions.enableInternet` 오류는 Internet 설정·권한을 확인해야 한다는 뜻입니다. [Kaggle 직원의 인증 안내](https://www.kaggle.com/product-feedback/202668)

Kaggle에서 위부터 실행합니다. 학습에는 GPU 한 장을 사용합니다. 첫 셀에서 Kaggle 환경과 실제 CUDA 연산을 확인합니다.

이 노트북은 fork의 고정 commit을 git clone/checkout하여 실행하며, 실행 중 소스를 수정하지 않습니다. 작은 랜덤 행동 데이터와 **1 epoch checkpoint**를 만듭니다. 학습·저장 연결 확인용이며, 논문 성능 재현이나 제어 실험 결과는 아닙니다. 아직 Kaggle GPU에서 실행 검증하지 않았습니다.

- 학습 입력: RGB history와 실제 적용된 action. 물리 상태는 데이터에 기록하지만 학습 loader가 사용하지 않습니다.
- train/calibration/evaluation은 서로 다른 episode seed로 생성합니다. calibration/evaluation 파일은 이번 학습에 사용하지 않습니다.
- 현재 범위는 world model 학습까지입니다. 보정 모델·안전 분류기·SLS MPC는 후속 단계입니다.
- 실행 결과는 Output에서 다운로드합니다. Quick Save만으로 checkpoint 백업이 끝났다고 판단하지 않습니다.
- Save & Run All은 별도의 새 세션에서 처음부터 다시 실행합니다.

[원본 코드](https://github.com/trustworthyrobotics/SLS-squared) · [논문 v1](https://arxiv.org/abs/2606.15594v1) · [Kaggle 사용 안내](https://www.kaggle.com/docs/notebooks)

## 1. 실행 설정
첫 실행은 아래 설정을 유지합니다. 본 학습은 새 RUN_NAME, 확정한 데이터량과 총 EPOCHS로 시작해야 합니다. 1 epoch 점검 checkpoint의 scheduler를 그대로 더 긴 학습에 재사용하지 않습니다.

완료된 epoch의 last.ckpt가 있을 때 재개할 수 있습니다. 이전 notebook output을 Add Input으로 연결하고 PREVIOUS_OUTPUT에 그 안의 run 폴더를 지정합니다. 같은 run은 데이터·batch·총 epoch 설정을 유지합니다. 첫 epoch 완료 전에는 재개 checkpoint가 없습니다.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shutil, time, platform
from importlib.metadata import version

assert Path("/kaggle/working").is_dir(), "이 노트북은 Kaggle에서 실행하세요. 로컬 학습을 시작하지 않습니다."
RUN_NAME = "reacher_setup_v2"
PREVIOUS_OUTPUT = None  # 예: "/kaggle/input/.../latent_geometric_control/reacher_setup_v1"
REPOSITORY_URL = "https://github.com/KwangBin417/SLS-squared.git"
UPSTREAM_COMMIT = "69bf9a8b779c1ec80e0ce7c0019ec82dfe5c4dbc"
COMMIT = "aa4e3d48c35b3a4b82d1c42f9da43af7fc599e5d"  # fork의 학습 코드 버전
SPLITS = {"train": {"episodes": 8, "seed": 1000},
          "calibration": {"episodes": 4, "seed": 2000},
          "evaluation": {"episodes": 4, "seed": 3000}}
STEPS_PER_EPISODE = 50
EPOCHS = 1
BATCH_SIZE = 4
WORKERS = 2
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
assert EPOCHS >= 1 and BATCH_SIZE >= 1
OUT = Path("/kaggle/working/latent_geometric_control") / RUN_NAME
if PREVIOUS_OUTPUT:
    previous = Path(PREVIOUS_OUTPUT)
    assert (previous / "run_config.json").is_file(), "이전 output의 run_config.json이 있는 폴더를 지정하세요."
    if not OUT.exists():
        shutil.copytree(previous, OUT)
OUT.mkdir(parents=True, exist_ok=True)
LOGS = OUT / "logs"
LOGS.mkdir(exist_ok=True)
SOURCE = Path("/kaggle/temp") / f"SLS-squared-{COMMIT}"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MPLCONFIGDIR"] = "/kaggle/temp/matplotlib"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "2"

def run(command, log_name, cwd=None):
    command = [str(x) for x in command]
    print("실행:", " ".join(command), flush=True)
    with (LOGS / log_name).open("a") as log:
        log.write("\nCOMMAND " + json.dumps(command) + "\n")
        with subprocess.Popen(command, cwd=cwd, env=os.environ.copy(),
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f"실행 실패. 확인할 로그: {LOGS / log_name}")

run(["nvidia-smi"], "gpu.txt")
import torch
assert torch.cuda.is_available(), "Session options → Accelerator에서 GPU를 선택하세요."
probe = torch.ones((16, 16), device="cuda")
assert (probe @ probe).sum().item() == 4096
del probe
torch.cuda.empty_cache()
capability = torch.cuda.get_device_capability(0)
PRECISION = "bf16-mixed" if capability[0] >= 8 else "16-mixed"
print("GPU:", torch.cuda.get_device_name(0), "| 학습 GPU 수: 1 | precision:", PRECISION)
CONFIG = {"repository_url": REPOSITORY_URL, "upstream_commit": UPSTREAM_COMMIT,
          "commit": COMMIT, "splits": SPLITS, "steps_per_episode": STEPS_PER_EPISODE,
          "epochs": EPOCHS, "batch_size": BATCH_SIZE, "workers": WORKERS,
          "precision": PRECISION, "train_seed": 3072,
          "image_size": 224, "embed_dim": 5, "history_size": 1, "num_preds": 5,
          "straighten_weight": 0.0, "action_noise_scale": 1.0, "control_hz": 50,
          "patch_version": "spt-0.1.8-manual-clip-step-schedule-v1"}
config_file = OUT / "run_config.json"
if config_file.exists():
    assert json.loads(config_file.read_text()) == CONFIG, "기존 run과 설정이 다릅니다. 새 RUN_NAME을 사용하세요."
else:
    config_file.write_text(json.dumps(CONFIG, indent=2))
print("결과 폴더:", OUT)

## 2. Fork의 고정 버전과 학습 패키지

Fork를 clone하고 위 COMMIT을 checkout합니다. 학습 코드는 이 저장소에서 관리하며, 이 셀은 내려받은 소스를 바꾸지 않습니다.

Kaggle의 기존 PyTorch·torchvision·NumPy 버전을 유지합니다. 전체 원본 requirements나 JAX/GPU-SLS는 이 단계에서 설치하지 않습니다. 패키지 설치가 충돌하면 로그를 확인하고 중단합니다.

In [ ]:
TEMP = Path("/kaggle/temp")
TEMP.mkdir(exist_ok=True)
if not SOURCE.exists():
    run(["git", "clone", "--no-checkout", REPOSITORY_URL, SOURCE], "source.txt")
    run(["git", "checkout", "--detach", COMMIT], "source.txt", SOURCE)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True).strip()
assert actual_commit == COMMIT, "Source commit이 설정과 다릅니다. 새 세션에서 실행하세요."
assert not subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"], cwd=SOURCE, text=True).strip(), "Source에 기록되지 않은 변경이 있습니다."
constraints = TEMP / "reacher_constraints.txt"
constraints.write_text("\n".join(f"{name}=={version(name)}" for name in ("torch", "torchvision", "numpy")) + "\n")
PACKAGES = ["dm_control==1.0.38", "stable-baselines3==2.7.1",
            "lightning==2.6.1", "stable-pretraining==0.1.8",
            "einops", "h5py", "imageio[ffmpeg]", "tqdm"]
run([sys.executable, "-m", "pip", "install", "--constraint", constraints, *PACKAGES], "install.txt")
assert (SOURCE / "reacher/train/mlpdyn_train.py").is_file()
os.environ["PYTHONPATH"] = str(SOURCE) + os.pathsep + os.environ.get("PYTHONPATH", "")
(OUT / "requirements-resolved.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
(OUT / "runtime.json").write_text(json.dumps({
    "python": sys.version, "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "torch_cuda": torch.version.cuda, "source_commit": COMMIT,
    "repository_url": REPOSITORY_URL, "upstream_commit": UPSTREAM_COMMIT,
}, indent=2))
run([sys.executable, "-c",
     "import torch, stable_pretraining as spt, lightning, dm_control, stable_baselines3; "
     "from stable_pretraining import Module; from reacher.shared.models import JEPA; "
     "encoder = spt.backbone.utils.vit_hf('tiny', patch_size=14, image_size=224, "
     "pretrained=False, use_mask_token=False); print('필수 import 및 ViT 구성 완료')"],
    "imports.txt", SOURCE)

## 3. 원본 대비 변경 기록

다음 셀은 fork에 이미 commit된 차이를 출력·저장합니다. 소스 파일을 수정하지 않습니다.

stable-pretraining 0.1.8에 맞춰 Reacher trainer 두 곳을 조정했습니다.

1. Manual optimization에서도 원본 gradient clipping을 유지하도록 SPT Manager와 같은 전달 방식을 사용합니다.
2. 해당 scheduler가 optimizer step 수로 길이를 계산하므로 interval을 epoch에서 step으로 변경했습니다. 원래 조합은 첫 epoch 내내 learning rate가 0일 수 있습니다.

이는 이 두 조정을 명시한 자체 학습 baseline이며, 저자 checkpoint의 성능 재현은 아닙니다. 원본 버전과 fork 버전, 변경 diff를 결과 폴더에 보존합니다.

In [ ]:
subprocess.check_call(
    ["git", "merge-base", "--is-ancestor", UPSTREAM_COMMIT, COMMIT], cwd=SOURCE)
patch = subprocess.check_output(
    ["git", "diff", UPSTREAM_COMMIT, COMMIT, "--", "reacher/train/mlpdyn_train.py"],
    cwd=SOURCE, text=True)
assert patch, "기대했던 trainer 호환성 변경이 없습니다."
(OUT / "training-compatibility.patch").write_text(patch)
(OUT / "source_provenance.json").write_text(json.dumps({
    "repository_url": REPOSITORY_URL,
    "commit": actual_commit,
    "upstream_repository": "https://github.com/trustworthyrobotics/SLS-squared.git",
    "upstream_commit": UPSTREAM_COMMIT,
    "trainer_sha256": hashlib.sha256(
        (SOURCE / "reacher/train/mlpdyn_train.py").read_bytes()).hexdigest(),
}, indent=2))
print(patch)

## 4. Reacher RGB 렌더 확인
팔 이미지가 나타나면 GPU 세션의 MuJoCo/EGL 렌더 경로가 동작한 것입니다.

In [ ]:
render_code = """
from pathlib import Path
from dm_control import suite
import imageio.v2 as imageio
import sys
env = suite.load("reacher", "hard", task_kwargs={"random": 123})
env.reset()
frame = env.physics.render(height=224, width=224, camera_id=0)
assert frame.shape == (224, 224, 3)
imageio.imwrite(Path(sys.argv[1]), frame)
print("RGB 렌더 완료", frame.shape)
env.close()
"""
run([sys.executable, "-c", render_code, OUT / "reacher_frame.png"], "render.txt", SOURCE)
from IPython.display import Image, display
display(Image(filename=str(OUT / "reacher_frame.png")))

## 5. 데이터 생성
총 16개의 짧은 episode를 생성합니다. 공식 random-action generator를 사용합니다. 저자의 expert-policy 데이터와는 다른 수집 방식입니다.

입력 scale을 1로 지정해 저장 명령과 실제 clip 후 입력을 맞추고, physics/control을 모두 50 Hz로 지정합니다. 데이터에 기록된 qpos/qvel은 진단용이며 학습에 들어가지 않습니다. 데이터가 이미 완성되어 있으면 다시 생성하지 않습니다.

In [ ]:
import h5py
import numpy as np
DATA = OUT / "data"
DATA.mkdir(exist_ok=True)
for split, spec in SPLITS.items():
    folder = DATA / split
    path = folder / f"{split}.h5"
    if path.exists():
        with h5py.File(path, "r") as h5:
            assert int(h5.attrs.get("num_episodes", -1)) == spec["episodes"], (
                f"미완성 데이터: {path}. 새 RUN_NAME으로 다시 시작하거나 해당 생성 결과를 확인하세요.")
        print("기존 데이터 사용:", path)
        continue
    run([sys.executable, "-m", "reacher.data.reacher_data_gen_random",
         "--task", "hard", "--outdir", folder, "--output-name", path.name,
         "--num-trajectories", spec["episodes"], "--trajectory-length", STEPS_PER_EPISODE,
         "--seed", spec["seed"], "--width", 224, "--height", 224,
         "--physics-freq-hz", 50, "--control-freq-hz", 50,
         "--action-noise-scale", 1.0, "--compression", "lzf"],
        f"generate-{split}.txt", SOURCE)

## 6. 학습 전에 데이터 확인
Episode 마지막 action은 NaN padding입니다. 나머지 action의 유효성·실제 범위, timestep, episode seed 분리를 확인하고 데이터 hash와 학습용 action 통계를 저장합니다.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

seed_sets, data_manifest = {}, {}
for split, spec in SPLITS.items():
    path = DATA / split / f"{split}.h5"
    with h5py.File(path, "r") as h5:
        lengths = h5["ep_len"][:].astype(int)
        offsets = h5["ep_offset"][:].astype(int)
        seeds = h5["episode_seed"][:].astype(int)
        actions = h5["action"][:]
        assert len(lengths) == spec["episodes"]
        assert np.array_equal(offsets, np.concatenate(([0], np.cumsum(lengths)[:-1])))
        assert len(actions) == int(lengths.sum()) == len(h5["pixels"])
        assert h5["pixels"].shape[1:] == (224, 224, 3)
        assert h5["pixels"].dtype == np.uint8
        assert np.all(lengths >= 7)
        assert np.isclose(h5.attrs["physics_timestep"], 0.02)
        assert np.isclose(h5.attrs["control_timestep"], 0.02)
        ends = offsets + lengths - 1
        valid_rows = np.ones(len(actions), dtype=bool)
        valid_rows[ends] = False
        assert np.isnan(actions[ends]).all()
        applied = actions[valid_rows]
        assert np.isfinite(applied).all() and np.max(np.abs(applied)) <= 1.0 + 1e-6
        seed_sets[split] = set(map(int, seeds))
        assert len(seed_sets[split]) == len(seeds)
        data_manifest[split] = {
            "path": str(path.relative_to(OUT)), "sha256": sha256_file(path),
            "episodes": len(lengths), "transitions": len(applied),
            "episode_seeds": seeds.tolist(), "file_bytes": path.stat().st_size,
        }
        if split == "train":
            assert int(np.maximum(lengths - 5, 0).sum()) // BATCH_SIZE >= 2
            stats = {"mean": applied.mean(axis=0).tolist(),
                     "std": np.maximum(applied.std(axis=0), 1e-6).tolist(),
                     "source_sha256": data_manifest[split]["sha256"]}
            (OUT / "action_stats_train.json").write_text(json.dumps(stats, indent=2))
for i, first in enumerate(seed_sets):
    for second in list(seed_sets)[i + 1:]:
        assert seed_sets[first].isdisjoint(seed_sets[second]), "episode seed가 겹칩니다."
manifest_file = OUT / "data_manifest.json"
if manifest_file.exists():
    assert json.loads(manifest_file.read_text()) == data_manifest, "기존 run의 데이터가 변경되었습니다. 새 RUN_NAME을 사용하세요."
else:
    manifest_file.write_text(json.dumps(data_manifest, indent=2))
print(json.dumps(data_manifest, indent=2))

## 7. 첫 world model 학습
이 셀은 실제로 GPU 학습을 시작합니다. 첫 실행은 1 epoch입니다.

논문 Reacher와 맞춘 기본 구조: latent 5차원, [현재 embedding, embedding 차분] 10차원 상태, 5-step prediction, MLP 512×2. Straightening은 weight 0으로 비활성화합니다. 작은 batch와 1 epoch는 실행 확인 설정입니다.

train 파일 전체로 학습하며 validation 성능은 산출하지 않습니다. 원본 train-split을 줄이면 같은 episode의 겹치는 window가 분리될 수 있으므로 여기서는 1.0을 유지합니다. calibration/evaluation 파일은 별도로 보관합니다.

완료된 epoch의 last.ckpt가 있으면 같은 설정으로 재실행해 재개할 수 있습니다. 첫 epoch 완료 전에 실패하면 로그의 원인을 해결하고 새 RUN_NAME으로 시작하십시오. 설정을 바꾸는 본 학습도 새 RUN_NAME으로 진행합니다.

In [ ]:
MODEL = OUT / "model"
completion_file = OUT / "training_complete.json"
command = [
    sys.executable, "-m", "reacher.train.mlpdyn_train",
    "--dataset-path", DATA / "train/train.h5", "--run-dir", MODEL,
    "--output-model-name", "lewm", "--seed", 3072, "--train-split", 1.0,
    "--img-size", 224, "--patch-size", 14, "--encoder-scale", "tiny",
    "--embed-dim", 5, "--history-size", 1, "--num-preds", 5, "--action-dim", 2,
    "--predictor-hidden-width", 512, "--predictor-depth", 2,
    "--straighten-weight", 0.0, "--epochs", EPOCHS, "--batch-size", BATCH_SIZE,
    "--num-workers", WORKERS, "--accelerator", "gpu", "--devices", 1,
    "--precision", PRECISION,
]
(OUT / "training_command.json").write_text(json.dumps([str(x) for x in command], indent=2))
if completion_file.exists():
    assert (MODEL / "last.ckpt").is_file()
    print("이 run의 학습은 이미 완료되었습니다. 재실행하지 않습니다.")
else:
    started = time.monotonic()
    # MODEL을 미리 생성하지 않습니다. 원본 trainer가 checkpoint 없는 기존 폴더를 거부합니다.
    run(command, "training.txt", SOURCE)
    checkpoint = MODEL / "last.ckpt"
    assert checkpoint.is_file(), "학습 로그에서 checkpoint 저장 여부를 확인하세요."
    # 이 notebook에서 직접 만든 checkpoint만 읽습니다.
    saved = torch.load(checkpoint, map_location="cpu", weights_only=False)
    assert int(saved.get("global_step", 0)) > 0
    assert all(torch.isfinite(value).all().item()
               for value in saved["state_dict"].values()
               if torch.is_tensor(value) and value.is_floating_point()), "checkpoint에 비유한 값이 있습니다."
    report = {"global_step": int(saved["global_step"]), "epoch": int(saved["epoch"]),
              "elapsed_this_session_seconds": time.monotonic() - started,
              "checkpoint_sha256": sha256_file(checkpoint),
              "meaning": "pipeline check only; no control or model-quality claim"}
    completion_file.write_text(json.dumps(report, indent=2))
    del saved
print(completion_file.read_text())

## 8. 결과 보관
Output에서 run 폴더 전체를 다운로드하거나 보존된 notebook output으로 저장합니다. 다음 세션에서는 Add Input으로 연결할 수 있습니다.

반드시 같이 보관할 파일: model/last.ckpt, model/config.json, data의 HDF5, run_config.json, data_manifest.json, action_stats_train.json, requirements-resolved.txt, source_provenance.json, training-compatibility.patch, logs.

Save & Run All은 새 세션에서 실행합니다. 이미 완료한 결과를 재사용하려면 먼저 보존한 output을 Add Input으로 연결하고 PREVIOUS_OUTPUT을 지정하십시오. Quick Save는 notebook 편집 저장이며 checkpoint 파일 백업 확인을 대신하지 않습니다.

다운로드용 zip을 만드는 다음 셀은 결과 복사본이어서 잠시 디스크 사용량이 늘어납니다. 첫 작은 run에만 사용하고, 대규모 데이터에서는 notebook output 보존을 사용하십시오. 사용을 마치면 Kaggle 세션을 Stop 합니다.

In [ ]:
total = sum(path.stat().st_size for path in OUT.rglob("*") if path.is_file())
print(f"현재 결과 크기: {total / 1024**2:.1f} MiB")
if total < 1024**3:
    archive = shutil.make_archive(
        str(OUT.parent / RUN_NAME), "zip", root_dir=OUT.parent, base_dir=RUN_NAME)
    print("Output에서 다운로드:", archive)
else:
    print("1 GiB 이상이므로 zip 중복 생성을 생략합니다. Output을 통해 보관하세요.")
print("학습을 마쳤으면 GPU 세션을 Stop 하세요.")